# AI Financial Risk Copilot: Risk Analytics & Behavioral Math Demo
### Research Sandbox for Explainable AI (XAI) and Portfolio Diversification Modeling

This Jupyter Notebook demonstrates the mathematical and statistical foundations of the **AI Financial Risk Copilot** framework. 

We implement and visualize the core statistical metrics discussed in the research paper:
1. **Herfindahl-Hirschman Index (HHI)** for concentration risk
2. **Diversification Health Score (DHS)**
3. **Annualized Portfolio Volatility ($\sigma_p$)** using asset covariance matrices
4. **Volatility Risk Factor (VRF)** benchmarked to market baseline
5. **Composite Investor Safety Score (ISS)** merging quantitative finance with behavioral sentiment analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set sleek aesthetic styling for matplotlib
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'ggplot')
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.family"] = "sans-serif"

## 1. Portfolio Diversification & Herfindahl-Hirschman Index (HHI)

The $HHI$ quantifies portfolio concentration risk:
$$HHI = \sum_{i=1}^{N} w_i^2$$

Where $w_i$ is the weight of asset $i$ as a decimal (summing to 1.0).

The **Diversification Health Score (DHS)** converts this into a beginner-friendly 0-100 gauge:
$$DHS = (1 - HHI) \times 100$$

In [ ]:
def calculate_dhs(weights):
    """
    Computes HHI and the Diversification Health Score (DHS).
    weights: list or numpy array of percentage weights (e.g. [35, 15, 40, 10])
    """
    # Convert to decimals summing to 1.0
    w_arr = np.array(weights) / 100.0
    w_arr = w_arr / np.sum(w_arr) # Ensure perfect sum to 1.0
    
    hhi = np.sum(w_arr ** 2)
    dhs = (1.0 - hhi) * 100.0
    return hhi, dhs

# Sample simulated portfolios (Tech, Speculative Meme, Crypto, Index)
portfolios = {
    "Perfectly Equal (4 Assets)": [25, 25, 25, 25],
    "Moderate Diversified (Our Sandbox)": [35, 15, 40, 10],
    "Speculative concentrated": [5, 5, 85, 5],
    "100% Single Asset (Extreme Concentration)": [0, 0, 100, 0]
}

print(f"{'Portfolio Profile':<45} | {'HHI':<8} | {'DHS Score':<10}")
print("-" * 70)
for name, w in portfolios.items():
    h, d = calculate_dhs(w)
    print(f"{name:<45} | {h:<8.4f} | {d:<10.2f}")

## 2. Annualized Portfolio Volatility ($\sigma_p$) via Covariance Matrix

Using annualized standard deviations for our four sandbox assets:
* Tech Stocks: $\sigma_1 = 18\%$
* Hype Stocks: $\sigma_2 = 75\%$
* Meme Crypto: $\sigma_3 = 95\%$
* Index Funds: $\sigma_4 = 15\%$

We model portfolio variance:
$$\sigma_p = \sqrt{\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}}$$
Assuming a baseline asset correlation of $\rho = 0.22$.

In [ ]:
def calculate_portfolio_volatility(weights, volatilities, correlation=0.22):
    """
    Calculates portfolio volatility using a constant correlation matrix.
    """
    w = np.array(weights) / 100.0
    w = w / np.sum(w) # Normalize
    
    n = len(weights)
    v = np.array(volatilities)
    
    # Construct Covariance Matrix
    cov_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i == j:
                cov_matrix[i, j] = v[i] ** 2
            else:
                cov_matrix[i, j] = correlation * v[i] * v[j]
                
    p_var = np.dot(w.T, np.dot(cov_matrix, w))
    p_vol = np.sqrt(p_var)
    return p_vol

vols = [0.18, 0.75, 0.95, 0.15] # Tech, Meme, Crypto, Index

print(f"{'Portfolio Profile':<45} | {'Volatility (Est)':<18}")
print("-" * 70)
for name, w in portfolios.items():
    pv = calculate_portfolio_volatility(w, vols)
    print(f"{name:<45} | {pv*100:<16.2f}%")

## 3. Visual Simulation: Diversification vs. Plunge Impact

We plot how shifting allocations from highly concentrated Meme Crypto/Stocks into broad, diversified indexes shields capital during a simulated Year 2 market crash (65% crypto crash vs 12% index dip).

In [ ]:
years = np.arange(0, 6)
capital_start = 10000

# 1. High Speculative Portfolio: 85% Crypto, 15% Meme (Vol: ~90%)
# 2. Safe Diversified Portfolio: 10% Tech, 10% Crypto, 80% Index (Vol: ~20%)

paths = {
    "Speculative Portfolio (Hype Heavy)": [10000, 14200, 4800, 5200, 7100, 9400],
    "Diversified Portfolio (Anchor Heavy)": [10000, 10700, 9400, 10100, 10900, 11700],
    "S&P 500 Baseline Growth": [10000, 10800, 11660, 12600, 13600, 14680]
}

for label, values in paths.items():
    marker = 'o' if 'Baseline' in label else 's'
    plt.plot(years, values, marker=marker, linewidth=2.5, label=label)

plt.title("5-Year Projection Simulator: Capital Plunge Scenarios", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Interaction Timeframe (Years)", fontsize=11)
plt.ylabel("Portfolio Valuation ($ USD)", fontsize=11)
plt.axvspan(1.8, 2.2, color='red', alpha=0.12, label='Market Crash Event')
plt.legend(loc="upper left", frameon=True, facecolor='#ffffff', framealpha=0.9)
plt.tight_layout()
plt.show()

## 4. Composite Investor Safety Score ($ISS$)

The $ISS$ brings qualitative behavioral biases and quantitative financial allocations together:
$$ISS = 100 - \left( 0.40 \cdot CR + 0.35 \cdot VRF + 0.25 \cdot \mathcal{B} \right)$$

Where:
* $CR$ (Concentration Risk) = $HHI \times 100$
* $VRF$ (Volatility Risk Factor) = $\min(100, (\sigma_p / 15\%) \times 50)$
* $\mathcal{B}$ (Behavioral Risk Score) = sentiment NLP output (25 to 100)

In [ ]:
def calculate_iss(weights, vols, b_score):
    hhi, dhs = calculate_dhs(weights)
    p_vol = calculate_portfolio_volatility(weights, vols)
    
    cr = hhi * 100.0
    vrf = min(100.0, (p_vol / 0.15) * 50.0)
    
    iss = 100 - (0.40 * cr + 0.35 * vrf + 0.25 * b_score)
    return round(iss), round(cr), round(vrf)

# Compare Investor States
print(f"{'Scenario & Mindset':<50} | {'ISS Score':<10} | {'Rating':<12}")
print("-" * 80)

# Case 1: Panic Retailer buying speculative Doge during a drop
score1, _, _ = calculate_iss([0, 90, 10, 0], vols, b_score=90) # high concentration, high fear
print(f"{'1. Revenge Speculator (Panic / Concentration)':<50} | {score1:<10} | {'Danger'}")

# Case 2: Broad balanced allocator with stable sentiment
score2, _, _ = calculate_iss([30, 10, 10, 50], vols, b_score=25) # high indexes, neutral state
print(f"{'2. Anchor Investor (Diversified / Rational)':<50} | {score2:<10} | {'Safe'}")

# Case 3: Overconfident leverage trader
score3, _, _ = calculate_iss([40, 60, 0, 0], vols, b_score=75) # moderate concentration, high hubris
print(f"{'3. Hubris Trader (Overconfidence / High-Beta)':<50} | {score3:<10} | {'Elevated Risk'}")